In [ ]:
# Required packages: numpy, h5py, and mne-python.
# If needed, install them in this notebook environment with:
# %pip install numpy h5py mne


In [ ]:
import copy
import os
from collections import Counter

import h5py
import mne
import numpy as np


In [ ]:
# Update these paths for the subject being converted.
subject_folder = r"//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s09"
subject_filename = "Subject9_spm_mne"
mat_filepath = os.path.join(subject_folder, f"{subject_filename}.mat")

# If there is exactly one .ds folder alongside the .mat file, it is found automatically.
# Set ctf_ds_path to the matching .ds folder explicitly if it is stored elsewhere or
# if the subject folder contains multiple .ds folders.
ctf_ds_path = None

if not os.path.isfile(mat_filepath):
    raise FileNotFoundError(f"MAT file not found: {mat_filepath}")

if ctf_ds_path is None:
    ds_candidates = [
        os.path.join(subject_folder, name)
        for name in os.listdir(subject_folder)
        if name.lower().endswith(".ds")
        and os.path.isdir(os.path.join(subject_folder, name))
    ]
    if len(ds_candidates) == 1:
        ctf_ds_path = ds_candidates[0]
    elif len(ds_candidates) == 0:
        raise FileNotFoundError(
            f"No .ds dataset found in {subject_folder}. "
            "Set ctf_ds_path to the matching CTF dataset folder."
        )
    else:
        raise ValueError(
            f"Found multiple .ds datasets in {subject_folder}: {ds_candidates}. "
            "Set ctf_ds_path to the one that produced this .mat file."
        )

if not os.path.isdir(ctf_ds_path):
    raise FileNotFoundError(f"CTF .ds dataset not found: {ctf_ds_path}")

print(f"MAT file: {mat_filepath}")
print(f"Metadata source: {ctf_ds_path}")


In [ ]:
# Load epoched signal values and channel names from the MATLAB v7.3 file.
with h5py.File(mat_filepath, "r") as f:
    print("MAT variables:", list(f.keys()))
    data = np.asarray(f["data"]).T

    # MNE expects (epochs, channels, time points).
    epochs_data = data.transpose(2, 0, 1)

    sfreq = float(np.asarray(f["fs"]).flatten()[0])
    chan_names = []
    for ref in f["chan_names"][:, 0]:
        char_codes = np.asarray(f[ref][()]).ravel()
        name = "".join(chr(int(code)) for code in char_codes if int(code) != 0).strip()
        chan_names.append(name)

if epochs_data.ndim != 3:
    raise ValueError(f"Expected 3D epoched data, got shape {epochs_data.shape}")
if epochs_data.shape[1] != len(chan_names):
    raise ValueError(
        f"MAT data contains {epochs_data.shape[1]} channels but has "
        f"{len(chan_names)} channel names"
    )
if len({name.casefold() for name in chan_names}) != len(chan_names):
    raise ValueError("MAT channel names are not unique, ignoring letter case")


In [ ]:
print(f"Data shape after transpose: {data.shape}")
print(f"Epoch data shape (epochs, channels, times): {epochs_data.shape}")
print(f"N channels: {len(chan_names)}")
print(f"Sampling rate: {sfreq:g} Hz")
print(f"Channel names: {chan_names}")


## Channel metadata\n\nThe epoched samples come from the `.mat` file. The corresponding CTF `.ds` recording supplies MNE's channel types and MEG sensor geometry. The conversion matches channels by name and stops if a scalp MEG channel cannot be matched or has no valid location, rather than saving it as `misc`.

In [ ]:
# Build MNE channel metadata from the matching CTF .ds recording.
# The .mat file supplies the epoched samples; the .ds file supplies the
# authoritative sensor types, coil definitions, positions, digitization points,
# and device-to-head transform.
ctf_raw = mne.io.read_raw_ctf(
    ctf_ds_path,
    preload=False,
    clean_names=True,
    verbose="ERROR",
)
ctf_info = ctf_raw.info

def _channel_key(name):
    return name.strip().casefold()

ctf_index_by_name = {}
for idx, name in enumerate(ctf_info["ch_names"]):
    key = _channel_key(name)
    if key in ctf_index_by_name:
        raise ValueError(f"Duplicate channel name in CTF dataset: {name}")
    ctf_index_by_name[key] = idx

meg_names = [name for name in chan_names if name.upper().startswith("M")]
if not meg_names:
    raise ValueError("No M-prefixed MEG sensor channels were found in the MAT file")

missing_meg = [
    name for name in meg_names
    if _channel_key(name) not in ctf_index_by_name
]
if missing_meg:
    raise ValueError(
        "These MEG channels are absent from the selected .ds dataset, so their "
        f"sensor metadata cannot be transferred: {missing_meg}"
    )

channel_types = []
matched_pairs = []
unmatched_other_channels = []

for target_idx, name in enumerate(chan_names):
    source_idx = ctf_index_by_name.get(_channel_key(name))
    if source_idx is None:
        unmatched_other_channels.append(name)
        if name.upper().startswith("UPPT"):
            channel_type = "stim"
        elif name.upper().startswith("EEG"):
            channel_type = "emg"
        else:
            channel_type = "misc"
    else:
        channel_type = mne.channel_type(ctf_info, source_idx)
        matched_pairs.append((target_idx, source_idx))

        # These channels carry the recorded EMG and button-trigger signals.
        if name.upper().startswith("EEG"):
            channel_type = "emg"
        elif name.upper().startswith("UPPT"):
            channel_type = "stim"

    if name.upper().startswith("M") and channel_type not in {"mag", "grad"}:
        raise ValueError(
            f"CTF channel {name} has type {channel_type!r}; expected 'mag' or "
            "'grad' for a scalp MEG sensor."
        )
    channel_types.append(channel_type)

info = mne.create_info(
    ch_names=chan_names,
    sfreq=sfreq,  # Keep the sampling rate of the epoched .mat data.
    ch_types=channel_types,
)

# Copy the CTF sensor geometry into the matching MAT channel slots.
with info._unlock():
    for target_idx, source_idx in matched_pairs:
        source_type = mne.channel_type(ctf_info, source_idx)
        if source_type in {"mag", "grad", "ref_meg"}:
            for key in ("loc", "coil_type", "coord_frame", "unit", "unit_mul"):
                info["chs"][target_idx][key] = copy.deepcopy(
                    ctf_info["chs"][source_idx][key]
                )

    info["dig"] = copy.deepcopy(ctf_info["dig"])
    info["dev_head_t"] = copy.deepcopy(ctf_info["dev_head_t"])

# Confirm every M-prefixed sensor has a recognized MEG type and a finite,
# non-zero 3D position before creating or saving the FIF file.
meg_indices = [
    idx for idx, name in enumerate(chan_names)
    if name.upper().startswith("M")
]
meg_types = [mne.channel_type(info, idx) for idx in meg_indices]
meg_locations = np.asarray(
    [info["chs"][idx]["loc"][:3] for idx in meg_indices],
    dtype=float,
)
bad_locations = (
    ~np.isfinite(meg_locations).all(axis=1)
    | (np.linalg.norm(meg_locations, axis=1) == 0)
)
if bad_locations.any():
    bad_names = [
        chan_names[meg_indices[idx]]
        for idx in np.flatnonzero(bad_locations)
    ]
    raise ValueError(
        "The supporting .ds file did not provide valid locations for these "
        f"MEG channels: {bad_names}"
    )

if unmatched_other_channels:
    print(
        "Warning: these non-MEG channels were not in the .ds file and use "
        f"name-based fallback types: {unmatched_other_channels}"
    )

print(f"Channel types: {Counter(channel_types)}")
print(
    f"MEG sensors with locations: {len(meg_indices)} "
    f"({Counter(meg_types)})"
)
print(f"MAT sampling rate retained: {info['sfreq']:g} Hz")
ctf_raw.close()


In [ ]:
# Create the epoched MNE object using the aligned signal and sensor metadata.
epochs = mne.EpochsArray(epochs_data, info, verbose=False)

print(epochs)
print("Epoch data shape:", epochs.get_data().shape)


In [ ]:
# Save the FIF file and read it back to confirm channel types and MEG locations survived.
outfile = os.path.join(subject_folder, f"{subject_filename}-epo.fif")
epochs.save(outfile, overwrite=True, verbose=False)

saved_epochs = mne.read_epochs(outfile, preload=False, verbose=False)
saved_meg_indices = [
    idx for idx, name in enumerate(saved_epochs.ch_names)
    if name.upper().startswith("M")
]
saved_meg_types = [
    mne.channel_type(saved_epochs.info, idx)
    for idx in saved_meg_indices
]
saved_meg_locations = np.asarray(
    [saved_epochs.info["chs"][idx]["loc"][:3] for idx in saved_meg_indices],
    dtype=float,
)

if saved_meg_types != meg_types:
    raise RuntimeError("MEG channel types changed while writing the FIF file")

# FIF stores channel locations as single-precision floats. Allow 1 micrometer
# for that serialization roundoff while still detecting meaningful geometry changes.
location_error_m = float(np.max(np.abs(saved_meg_locations - meg_locations)))
location_tolerance_m = 1e-6
if not np.allclose(
    saved_meg_locations,
    meg_locations,
    rtol=0,
    atol=location_tolerance_m,
):
    raise RuntimeError(
        "MEG sensor locations differ after writing the FIF file: "
        f"maximum component difference was {location_error_m * 1e6:.3f} µm "
        f"(tolerance {location_tolerance_m * 1e6:.3f} µm)"
    )

print(f"Saved FIF: {outfile}")
print(f"Verified {len(saved_meg_indices)} MEG channels and their locations.")
print(f"Maximum FIF location roundoff: {location_error_m * 1e6:.3f} µm")
